In [3]:
import os
import requests
from bs4 import BeautifulSoup
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentType, initialize_agent
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
os.environ["TAVILY_API_KEY"]= "tavily_key"
os.environ["GOOGLE_API_KEY"]= "google_key"

In [5]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)
tavily_tool = TavilySearchResults(
    max_results=5
)

In [6]:
@tool
def scrape_page(url: str) -> str:
    """
    Read webpage content from a URL.
    Useful when review pages, blogs, articles,
    competitor pages or customer feedback pages
    need deeper analysis.
    """

    try:

        headers = {
            "User-Agent":
            "Mozilla/5.0"
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        return text[:5000]

    except Exception as e:

        return f"Error scraping page: {str(e)}"

In [7]:
tools = [
    tavily_tool,
    scrape_page
]

In [9]:
prompt = PromptTemplate.from_template(
"""
You are an expert Market Research Agent.

Your job is to help founders understand a market.

You should:

1. Understand the product idea.
2. Identify major competitors.
3. Search for competitor reviews.
4. Search for customer complaints.
5. Search for feature requests.
6. Identify market gaps.
7. Recommend MVP features.
8. Generate a SWOT analysis.

Always use Tavily Search first.

If URLs are found that contain reviews,
customer feedback, product comparisons,
or competitor analysis, use scrape_page
to read them.

Available Tools:

{tools}

Use this format:

Question: the input question

Thought: think about what to do

Action: one of [{tool_names}]

Action Input: input to tool

Observation: result

... repeat as needed ...

Thought: I now know the final answer

Final Answer:
Comprehensive Market Research Report


Question: {input}

{agent_scratchpad}
"""
)

In [10]:
agent = initialize_agent(
    llm=model,
    tools=tools,
    prompt=prompt,
    verbose=True,
    handle_parsing_errors=True
)

C:\Users\hp\AppData\Local\Temp\ipykernel_14276\938170513.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [11]:
for chunk in agent.stream("i want to start a cement company in india , what is competitor scenario in india ?"):
    print(chunk)



> Entering new None chain...
{'actions': [AgentAction(tool='tavily_search_results_json', tool_input='competitive landscape of cement industry in India 2024 2025', log='Question: i want to start a cement company in india , what is competitor scenario in india ?\n\nThought: I need to research the current competitive landscape of the Indian cement industry, identify the major players, understand market concentration, and look for trends or barriers to entry.\n\nAction: tavily_search_results_json\nAction Input: "competitive landscape of cement industry in India 2024 2025"')], 'messages': [AIMessage(content='Question: i want to start a cement company in india , what is competitor scenario in india ?\n\nThought: I need to research the current competitive landscape of the Indian cement industry, identify the major players, understand market concentration, and look for trends or barriers to entry.\n\nAction: tavily_search_results_json\nAction Input: "competitive landscape of cement industry 